#**Caso integrador: Evaluación de una intervención clínica**


**Contexto**
En un hospital de segundo nivel se implementó una intervención clínica con el objetivo de mejorar el estado hematológico de los pacientes hospitalizados.
Se recolectaron datos de pacientes adultos que fueron seguidos durante su estancia hospitalaria. Para cada paciente se registraron variables clínicas y de laboratorio antes y después de la intervención, así como la ocurrencia de complicaciones.
Los pacientes fueron clasificados en distintos grupos de tratamiento, y se cuenta además con información relevante como:

**Edad y sexo**
Índice de masa corporal (IMC)
Presencia de diabetes e hipertensión
Niveles de hemoglobina antes y después de la intervención
Días de estancia hospitalaria
Ocurrencia de alguna complicación clínica


** Objetivo general**
Utilizar herramientas de análisis de datos en R para evaluar:

El efecto de la intervención sobre la hemoglobina
La asociación entre comorbilidades y complicaciones
Diferencias en la estancia hospitalaria entre grupos

In [ ]:

library(tidyverse)
library(janitor)
library(gtsummary)
library(broom)

dir.create("resultados", showWarnings = FALSE)
dir.create("figuras", showWarnings = FALSE)

In [ ]:
# ----------------------------------------------------------
# 1. Cargar y preparar datos
# ----------------------------------------------------------

datos <- read_csv("data/datos_medicos_demo.csv") %>%
  clean_names() %>%
  mutate(
    sexo = factor(sexo),
    grupo = factor(grupo),
    fumador = factor(fumador),
    diabetes = factor(diabetes, levels = c(0, 1), labels = c("No", "Sí")),
    hipertension = factor(hipertension, levels = c(0, 1), labels = c("No", "Sí")),
    evento_complicacion = factor(evento_complicacion, levels = c(0, 1), labels = c("No", "Sí")),
    obesidad = case_when(
      is.na(imc) ~ NA_character_,
      imc >= 30 ~ "Sí",
      TRUE ~ "No"
    ),
    obesidad = factor(obesidad, levels = c("No", "Sí")),
    cambio_hb = hb_post - hb_pre
  )

#**Preguntas clínicas**
🔹 1. Cambio en hemoglobina
¿La intervención se asocia con un cambio significativo en los niveles de hemoglobina?

Compare los valores pre y post




> Evalúe significancia estadística
Interprete relevancia clínica

In [ ]:
# ----------------------------------------------------------
# ¿La hemoglobina cambió después de la intervención?
# ----------------------------------------------------------

resumen_hb <- datos %>%
  summarise(
    n = n(),
    hb_pre_media = mean(hb_pre, na.rm = TRUE),
    hb_pre_de = sd(hb_pre, na.rm = TRUE),
    hb_post_media = mean(hb_post, na.rm = TRUE),
    hb_post_de = sd(hb_post, na.rm = TRUE),
    cambio_medio = mean(cambio_hb, na.rm = TRUE),
    cambio_de = sd(cambio_hb, na.rm = TRUE)
  )

resumen_hb

prueba_hb <- t.test(datos$hb_post, datos$hb_pre, paired = TRUE)
tidy(prueba_hb)

# Gráfico pre-post resumido
hb_largo <- datos %>%
  select(id, hb_pre, hb_post) %>%
  pivot_longer(cols = c(hb_pre, hb_post), names_to = "momento", values_to = "hb") %>%
  mutate(momento = factor(momento, levels = c("hb_pre", "hb_post"), labels = c("Pre", "Post")))

g_hb <- ggplot(hb_largo, aes(x = momento, y = hb)) +
  geom_boxplot() +
  labs(
    title = "Hemoglobina antes y después",
    x = "Momento",
    y = "Hemoglobina"
  ) +
  theme_minimal()

g_hb
ggsave("figuras/caso_hb_pre_post.png", g_hb, width = 7, height = 5, dpi = 300)

#**2. Diabetes y complicaciones**

¿La presencia de diabetes aumenta el riesgo de complicaciones?

Analice la asociación entre diabetes y evento de complicación
Estime el riesgo en ambos grupos
Calcule razones de riesgo (RR)

In [ ]:
# ----------------------------------------------------------
# ¿La diabetes se asocia con complicación?
# ----------------------------------------------------------

tabla <- table(datos$diabetes, datos$evento_complicacion)
tabla

prueba_chi <- chisq.test(tabla)
prueba_chi
tidy(prueba_chi)

# Riesgo de complicación por diabetes
riesgos <- datos %>%
  group_by(diabetes) %>%
  summarise(
    n = n(),
    complicaciones = sum(evento_complicacion == "Sí"),
    riesgo = complicaciones / n
  )

riesgos

riesgo_diabetes <- riesgos$riesgo[riesgos$diabetes == "Sí"]
riesgo_no_diabetes <- riesgos$riesgo[riesgos$diabetes == "No"]
rr_diabetes <- riesgo_diabetes / riesgo_no_diabetes
diferencia_riesgo <- riesgo_diabetes - riesgo_no_diabetes

rr_diabetes
diferencia_riesgo

#**3. Estancia hospitalaria**


¿Existen diferencias en los días de estancia entre los grupos de tratamiento?

Compare distribuciones entre grupos
Justifique la prueba estadística utilizada
Interprete los resultados

In [ ]:
# ----------------------------------------------------------
# 4. Pregunta clínica 3
# ¿Los días de estancia difieren entre grupos?
# ----------------------------------------------------------

datos %>%
  group_by(grupo) %>%
  summarise(
    n = n(),
    mediana = median(dias_estancia, na.rm = TRUE),
    q1 = quantile(dias_estancia, 0.25, na.rm = TRUE),
    q3 = quantile(dias_estancia, 0.75, na.rm = TRUE)
  )

kruskal.test(dias_estancia ~ grupo, data = datos)

g_estancia <- ggplot(datos, aes(x = grupo, y = dias_estancia)) +
  geom_boxplot() +
  labs(
    title = "Días de estancia por grupo",
    x = "Grupo",
    y = "Días de estancia"
  ) +
  theme_minimal()

g_estancia
ggsave("figuras/caso_dias_estancia_grupo.png", g_estancia, width = 7, height = 5, dpi = 300)


In [ ]:
# ----------------------------------------------------------
# Tabla clínica final
# ----------------------------------------------------------

tabla_final <- datos %>%
  select(edad, sexo, grupo, imc, diabetes, hipertension, glucosa_mg_dl, dias_estancia, evento_complicacion) %>%
  tbl_summary(
    by = evento_complicacion,
    statistic = list(
      all_continuous() ~ "{median} [{p25}, {p75}]",
      all_categorical() ~ "{n} ({p}%)"
    ),
    missing = "ifany"
  ) %>%
  add_overall() %>%
  add_p()

tabla_final


In [ ]:
# ----------------------------------------------------------
# Guardar resultados principales
# ----------------------------------------------------------

write_csv(resumen_hb, "resultados/resumen_hb.csv")
write_csv(riesgos, "resultados/riesgo_complicacion_diabetes.csv")


In [ ]:
# ----------------------------------------------------------
# 7. Plantilla de interpretación
# ----------------------------------------------------------

cat("
INTERPRETACIÓN SUGERIDA:

1. Hemoglobina:
Se observó un cambio medio de hemoglobina de ",
round(resumen_hb$cambio_medio, 2),
" unidades. La prueba t pareada produjo un p valor de ",
round(prueba_hb$p.value, 4),
". La interpretación debe considerar además el intervalo de confianza y la relevancia clínica.

2. Diabetes y complicación:
El riesgo de complicación en pacientes con diabetes fue de ",
round(riesgo_diabetes, 3),
", mientras que en pacientes sin diabetes fue de ",
round(riesgo_no_diabetes, 3),
". El RR estimado fue ",
round(rr_diabetes, 2),
".

3. Estancia:
La comparación de días de estancia entre grupos se evaluó con Kruskal-Wallis por tratarse de una variable discreta/asimétrica.
")
